# 04 · Audit kalender ekonomi dan fitur makro (K3)

**Bab 3 Subbab 3.6.2 dan 3.7.3.**

**Gerbang K3** — K3 hanya dijalankan bila:
1. berkas kalender sama dengan hash SHA-256 arsip beku;
2. kolom waktu UTC baru (tanpa menimpa waktu mentah) cocok tepat dengan **10 jangkar** rilis resmi lintas DST.

**Asumsi operasional:** kolom `forecast` MT5 diperlakukan sebagai prakiraan yang diketahui sebelum `actual`.
Riwayat revisi independen tidak tersedia, jadi hasil K3 berlaku untuk snapshot ini dan **bukan** simulasi
*point-in-time* penuh.

**Input:** kalender MT5 · **Output:** `04_fitur_makro.parquet`

In [1]:
import tomllib
from pathlib import Path

# Cari folder proyek (folder yang memuat configs/phase4-p0.toml) agar notebook bisa dijalankan dari mana saja.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/phase4-p0.toml").exists())
CFG = tomllib.loads((ROOT / "configs/phase4-p0.toml").read_text(encoding="utf-8"))
OUT = ROOT / "data/interim/phase4-p0"     # folder hasil; diabaikan Git karena memuat data vendor
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
import hashlib

import pandas as pd

M = CFG["makro"]
path = ROOT / CFG["inputs"]["kalender"]["path"]
kal = pd.read_csv(path)

gerbang = {
    "hash SHA-256 sama dengan arsip beku": hashlib.sha256(path.read_bytes()).hexdigest() == CFG["inputs"]["kalender"]["sha256"],
    "value_id tidak duplikat": not kal["value_id"].duplicated().any(),
    "server MetaQuotes-Demo, build 6104": bool((kal["server_name"] == "MetaQuotes-Demo").all() and (kal["terminal_build"] == 6104).all()),
}
kal["event_time_utc"] = pd.to_datetime(kal["event_time_server"], format="%Y.%m.%dT%H:%M:%S", utc=True)
print("Jumlah kejadian:", len(kal))
pd.Series(gerbang, name="lulus").to_frame()

Jumlah kejadian: 706


,lulus
hash SHA-256 sama dengan arsip beku,True
value_id tidak duplikat,True
"server MetaQuotes-Demo, build 6104",True


## 1. Sepuluh jangkar waktu rilis resmi

Waktu rilis resmi BLS (CPI, NFP) dan Federal Reserve (FOMC) dipilih pada periode DST dan non-DST. Bila waktu mentah
kalender dibaca sebagai UTC, semuanya harus cocok tepat.

In [3]:
jangkar = pd.DataFrame(M["anchors_utc"], columns=["event_code", "waktu_resmi_utc"])
jangkar["waktu_resmi_utc"] = pd.to_datetime(jangkar["waktu_resmi_utc"], utc=True)
jangkar["cocok"] = [((kal["event_code"] == k) & (kal["event_time_utc"] == t)).sum() == 1
                    for k, t in zip(jangkar["event_code"], jangkar["waktu_resmi_utc"])]
gerbang["10 jangkar UTC cocok"] = bool(jangkar["cocok"].all())
STATUS_K3 = "LAYAK" if all(gerbang.values()) else "NOT TESTED"
print("Status gerbang K3:", STATUS_K3)
jangkar

Status gerbang K3: LAYAK


,event_code,waktu_resmi_utc,cocok
0,US_NFP,2020-01-10 13:30:00+00:00,True
1,US_CPI_MM,2020-01-14 13:30:00+00:00,True
2,US_NFP,2020-07-02 12:30:00+00:00,True
3,US_CPI_MM,2020-07-14 12:30:00+00:00,True
4,US_NFP,2020-11-06 13:30:00+00:00,True
5,US_CPI_MM,2020-11-12 13:30:00+00:00,True
6,US_FOMC,2024-01-31 19:00:00+00:00,True
7,US_NFP,2024-07-05 12:30:00+00:00,True
8,US_CPI_MM,2024-07-11 12:30:00+00:00,True
9,US_FOMC,2024-07-31 18:00:00+00:00,True


## 2. Isi kalender per keluarga kejadian

In [4]:
ringkas = kal.groupby("event_code").agg(
    kejadian=("value_id", "size"),
    actual_ada=("actual", lambda s: s.notna().sum()),
    forecast_ada=("forecast", lambda s: s.notna().sum()),
    pertama=("event_time_utc", "min"),
    terakhir=("event_time_utc", "max"),
)
ringkas

,kejadian,actual_ada,forecast_ada,pertama,terakhir
event_code,,,,,
US_CORE_CPI_MM,77,77,75,2020-01-14 13:30:00+00:00,2026-07-14 12:30:00+00:00
US_CORE_CPI_YY,78,78,77,2020-01-14 13:30:00+00:00,2026-07-14 12:30:00+00:00
US_CPI_MM,77,77,75,2020-01-14 13:30:00+00:00,2026-07-14 12:30:00+00:00
US_CPI_YY,78,78,77,2020-01-14 13:30:00+00:00,2026-07-14 12:30:00+00:00
US_FOMC,186,0,0,2020-01-03 19:00:00+00:00,2026-07-29 18:30:00+00:00
US_NFP,78,78,78,2020-01-10 13:30:00+00:00,2026-07-02 12:30:00+00:00
US_RATE,54,54,0,2020-01-29 19:00:00+00:00,2026-07-29 18:00:00+00:00
US_UNEMPLOYMENT,78,78,78,2020-01-10 13:30:00+00:00,2026-07-02 12:30:00+00:00


`US_RATE` dan `US_FOMC` tidak punya forecast, jadi tidak diberi angka kejutan (Bab 3); keduanya hanya menyumbang
umur rilis dan penanda waktu.

### Kejadian dengan waktu sama

Bab 3 meminta urutan tetap bila beberapa pengumuman punya waktu sama. Hanya FOMC yang mengalaminya (pernyataan dan
proyeksi ekonomi pada menit yang sama). Karena FOMC tidak punya angka, satu baris per waktu sudah mewakili seluruh
informasinya.

In [5]:
sama = kal[kal.duplicated(["event_code", "event_time_utc"], keep=False)]
print(sama.groupby("event_code").size().rename("baris dengan waktu sama").to_string())
assert set(sama["event_code"]) <= set(M["flag_only_codes"]), "Keluarga numerik punya kejadian berwaktu sama"

event_code
US_FOMC    50


## 3. Bentuk fitur makro

Untuk setiap keluarga dan setiap waktu keputusan $\tau_t$:

- ambil rilis **terakhir yang waktunya lebih kecil secara ketat** dari $\tau_t$ (rilis tepat pada $\tau_t$ ditunda);
- `surprise` = actual − forecast (enam keluarga numerik saja);
- `age_hours` = umur rilis dalam jam, dibatasi 168; pada umur ≥ 168 jam `surprise` dikosongkan dan `expired` = 1;
- `within_1h`, `within_6h`, `within_24h` = 1 bila rilis terjadi dalam rentang itu.

Nilai kosong **tidak** diubah menjadi nol; imputasi dilakukan nanti dari data latih (notebook 06).

In [6]:
waktu = pd.read_parquet(OUT / "02_label.parquet", columns=["time", "decision_time"])
kiri = waktu.reset_index()
makro = pd.DataFrame({"time": waktu["time"]})

for kode in M["numeric_codes"] + M["flag_only_codes"]:
    p = f"macro_{kode.lower()}"
    ev = (kal[kal["event_code"] == kode]
          .sort_values(["event_time_utc", "value_id"])
          .drop_duplicates("event_time_utc", keep="last"))
    ev = ev.assign(surprise=pd.to_numeric(ev["actual"], errors="coerce") - pd.to_numeric(ev["forecast"], errors="coerce"))
    j = pd.merge_asof(kiri, ev[["event_time_utc", "surprise"]], left_on="decision_time", right_on="event_time_utc",
                      direction="backward", allow_exact_matches=False).set_index("index")
    umur = (j["decision_time"] - j["event_time_utc"]).dt.total_seconds() / 3600
    kedaluwarsa = umur >= M["age_cap_hours"]
    if kode in M["numeric_codes"]:
        makro[f"{p}_surprise"] = j["surprise"].where(~kedaluwarsa)
    makro[f"{p}_age_hours"] = umur.clip(upper=M["age_cap_hours"])
    makro[f"{p}_expired"] = kedaluwarsa.astype(int)
    for jam in M["within_hours"]:
        makro[f"{p}_within_{jam}h"] = ((umur > 0) & (umur <= jam)).astype(int)

MAKRO = [c for c in makro.columns if c.startswith("macro_")]
print("Kolom makro:", len(MAKRO))

Kolom makro: 46


### Contoh: rilis NFP 5 Juli 2024 pukul 12:30 UTC

Keputusan pukul 12:00 belum melihat rilis. Keputusan 13:00 sudah melihatnya (umur 0,5 jam), dan seterusnya.

In [7]:
contoh = makro.merge(waktu, on="time")
contoh = contoh[(contoh["decision_time"] >= "2024-07-05 11:00") & (contoh["decision_time"] <= "2024-07-05 16:00")]
contoh[["decision_time", "macro_us_nfp_surprise", "macro_us_nfp_age_hours", "macro_us_nfp_within_1h", "macro_us_nfp_within_6h"]]

,decision_time,macro_us_nfp_surprise,macro_us_nfp_age_hours,macro_us_nfp_within_1h,macro_us_nfp_within_6h
26650,2024-07-05 11:00:00+00:00,NaN,168.0,0,0
26651,2024-07-05 12:00:00+00:00,NaN,168.0,0,0
26652,2024-07-05 13:00:00+00:00,-81.0,0.5,1,1
26653,2024-07-05 14:00:00+00:00,-81.0,1.5,0,1
26654,2024-07-05 15:00:00+00:00,-81.0,2.5,0,1
26655,2024-07-05 16:00:00+00:00,-81.0,3.5,0,1


In [8]:
makro.to_parquet(OUT / "04_fitur_makro.parquet", index=False)
(OUT / "04_status_k3.txt").write_text(STATUS_K3 + "\n", encoding="utf-8")
print("Tersimpan:", (OUT / "04_fitur_makro.parquet").relative_to(ROOT), makro.shape, "· status K3:", STATUS_K3)

Tersimpan: data/interim/phase4-p0/04_fitur_makro.parquet (38799, 47) · status K3: LAYAK
